# Woche 6 · Vom Basismodell zum Assistenten: Prompting und Structured Output
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Letzte Woche hat GPT-2 eine Anweisung einfach fortgesetzt. Assistenten wie Mistral Small 4 befolgen sie, weil sie nach dem Pretraining weiter trainiert wurden. Heute nutzt ihr das gezielt: Ihr baut einen Klassifikator für Störungsmeldungen und verbessert ihn in fünf Stufen, **gemessen** an 20 gelabelten Meldungen.

| Stufe | Technik |
|---|---|
| 1 | naiver Prompt |
| 2 | klarer System-Prompt mit Definitionen und JSON-Format |
| 3 | Few-Shot: Beispiele im Prompt |
| 4 | Structured Output: Schema wird garantiert |
| 5 | Structured Output mit Überlegung zuerst |

**Lernziele**
- Prompts strukturiert schreiben: Rolle, Kontext, Aufgabe, Definitionen, Format, Beispiele
- Structured Output per Schema erzwingen und mit Pydantic validieren
- Eine Prompt-Variante gegen ein Testset messen und die Unsicherheit angeben
- Nutzen und Aufwand einer vorangestellten Überlegung abwägen

Pflicht: TODO 1–6, Ziel: TODO 7–8. API-Key als Colab-Secret nötig. Kosten: keine, die KI-Toolbox ist kostenlos (Fair Use).
In den letzten 15 Minuten bildet ihr eure Projektteams (siehe Ende des Notebooks).

In [ ]:
%pip install -q openai==3.24.0

In [ ]:
import json, math, re, time
from typing import Literal
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

def check(name, test):
    """Selbsttest: zeigt ✅ oder ❌. Fehler im Test gelten als 'noch nicht gelöst'."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

try:
    from google.colab import userdata
    API_KEY = userdata.get("KIT_TOOLBOX_KEY")
except ImportError:
    import os
    API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
API_KEY = API_KEY.strip() if API_KEY else None

BASE_URL = "https://ki-toolbox.scc.kit.edu/api/v1"              # KI-Toolbox des KIT
client = OpenAI(api_key=API_KEY or "fehlt", base_url=BASE_URL, max_retries=5)   # mehr Wiederholungen, falls viele gleichzeitig rechnen
MODEL = "kit.mistral-small-4-119b-a8b"
PREIS_INPUT_USD, PREIS_OUTPUT_USD = 1.00, 5.00                  # kommerzielle Referenz pro 1 Mio. Tokens (Woche 2)

def kosten_cent(usage):
    """Was der Aufruf bei einem kommerziellen Anbieter kosten würde (Referenzpreis). In der KI-Toolbox zahlt ihr nichts."""
    return (usage.prompt_tokens * PREIS_INPUT_USD + usage.completion_tokens * PREIS_OUTPUT_USD) / 1_000_000 * 100

def antworttext(antwort):
    """Text der Antwort."""
    return antwort.choices[0].message.content or ""

print("Schlüssel geladen:", bool(API_KEY), "| Modell:", MODEL)

---
# Teil A · Daten und Messung

## A1 · Testset: 20 gelabelte Störungsmeldungen

Jede Meldung hat drei Labels:
- **kategorie:** `mechanisch`, `elektrisch`, `hydraulik_kuehlung` oder `steuerung_software`
- **prioritaet:** `hoch` (Maschine steht oder Sicherheitsrisiko), `mittel` (Maschine läuft eingeschränkt, Qualität oder Tempo leiden), `niedrig` (kein Einfluss auf die laufende Produktion)
- **maschine:** die ID aus der Maschinenliste

In [ ]:
MASCHINEN = {"M-01": "CNC-Fräszentrum 1", "M-02": "CNC-Fräszentrum 2", "M-03": "CNC-Drehmaschine", "M-04": "Rundschleifmaschine",
             "M-05": "Roboter-Schweißzelle", "M-06": "Montagelinie A", "M-07": "Prüfstand", "M-08": "Teilewaschanlage"}

def m(text, kategorie, prioritaet, maschine):
    return {"text": text, "kategorie": kategorie, "prioritaet": prioritaet, "maschine": maschine}

TESTSET = [
    m("Die Spindel der Drehmaschine vibriert stark, Teile außerhalb der Toleranz, wir fahren mit halber Drehzahl weiter.", "mechanisch", "mittel", "M-03"),
    m("Schweißroboter steht, die Schweißstromquelle zeigt Fehler E21.", "elektrisch", "hoch", "M-05"),
    m("Kühlmittel tropft aus einer Leitung am Fräszentrum 1. Maschine läuft normal, Wanne ist untergestellt.", "hydraulik_kuehlung", "niedrig", "M-01"),
    m("Prüfstand: Messprotokolle werden nicht gespeichert, die Prüfung selbst funktioniert, Werte müssen von Hand notiert werden.", "steuerung_software", "mittel", "M-07"),
    m("Not-Aus an der Montagelinie A lässt sich nicht mehr zurücksetzen, die Linie steht.", "elektrisch", "hoch", "M-06"),
    m("Werkzeugwechsler am Fräszentrum 2 hakt gelegentlich, ein Wechsel dauert doppelt so lang.", "mechanisch", "mittel", "M-02"),
    m("Nach dem Update zeigt die Rundschleifmaschine die Bedienoberfläche auf Englisch an.", "steuerung_software", "niedrig", "M-04"),
    m("Hydraulikdruck an der Schweißzelle fällt ab, die Spannvorrichtung öffnet sich während des Schweißens. Verletzungsgefahr!", "hydraulik_kuehlung", "hoch", "M-05"),
    m("Förderband der Teilewaschanlage gerissen, es kommen keine Teile mehr durch.", "mechanisch", "hoch", "M-08"),
    m("Am Prüfstand ist die Prüfplakette der elektrischen Sicherheitsprüfung abgelaufen.", "elektrisch", "niedrig", "M-07"),
    m("Drehmaschine: Der Kühlschmierstoff schäumt stark, die Oberflächengüte leidet.", "hydraulik_kuehlung", "mittel", "M-03"),
    m("Die SPS der Montagelinie A hat die Verbindung verloren, die Linie steht seit 20 Minuten.", "steuerung_software", "hoch", "M-06"),
    m("Schleifscheibe der Rundschleifmaschine zeigt Risse, Maschine sofort gestoppt.", "mechanisch", "hoch", "M-04"),
    m("Heizstab der Waschanlage defekt, die Sicherung fliegt raus. Wasser erreicht nur 40 statt 60 Grad, Teile bleiben leicht fettig.", "elektrisch", "mittel", "M-08"),
    m("Fräszentrum 1: Das CNC-Programm für Gehäuse B lässt sich nicht laden, andere Programme laufen.", "steuerung_software", "mittel", "M-01"),
    m("Lichtschranke an der Zuführung der Montagelinie wird durch Staub ausgelöst, immer wieder kurze Unterbrechungen.", "elektrisch", "mittel", "M-06"),
    m("Leichtes Quietschen an der Z-Achse des Fräszentrums 2, keine Auswirkung auf die Teile.", "mechanisch", "niedrig", "M-02"),
    m("Die Druckluft am Prüfstand schwankt, Messungen müssen wiederholt werden.", "hydraulik_kuehlung", "mittel", "M-07"),
    m("Der Bildschirm am Bedienpult der Schweißzelle flackert, die Bedienung ist aber möglich.", "elektrisch", "niedrig", "M-05"),
    m("Ölstand im Hydraulikaggregat der Drehmaschine unter Minimum, die Maschine hat sich selbst abgeschaltet.", "hydraulik_kuehlung", "hoch", "M-03"),
]
BEISPIELE = [   # für Few-Shot, bewusst NICHT im Testset
    m("Achsantrieb X des Fräszentrums 1 meldet Überstrom, Maschine steht.", "elektrisch", "hoch", "M-01"),
    m("Filter im Kühlkreislauf der Rundschleifmaschine fast voll, Durchfluss noch ausreichend.", "hydraulik_kuehlung", "niedrig", "M-04"),
    m("Teilewaschanlage: Zeitsteuerung springt nach Stromausfall auf Werkseinstellung, Waschzyklen zu kurz.", "steuerung_software", "mittel", "M-08"),
]
FELDER = ["kategorie", "prioritaet", "maschine"]
print(len(TESTSET), "Testfälle,", len(BEISPIELE), "Beispiele")

### TODO 1 · Genauigkeit eines Feldes
`genauigkeit(vorhersagen, gold, feld)` gibt den Anteil der Fälle zurück, in denen `vorhersagen[i][feld] == gold[i][feld]`. Eine Vorhersage kann `None` sein (Antwort nicht lesbar). Dann zählt der Fall als falsch.

In [ ]:
def genauigkeit(vorhersagen, gold, feld):
    # TODO 1
    pass

In [ ]:
_g = [{"kategorie": "a"}, {"kategorie": "b"}, {"kategorie": "c"}, {"kategorie": "d"}]
_v = [{"kategorie": "a"}, None, {"kategorie": "c"}, {"kategorie": "x"}]
check("TODO 1a Anteil richtig", lambda: math.isclose(genauigkeit(_v, _g, "kategorie"), 0.5))
check("TODO 1b alles richtig", lambda: genauigkeit(_g, _g, "kategorie") == 1.0)

### TODO 2 · Wie genau ist „genau“?
20 Testfälle sind wenige. Eine gemessene Genauigkeit $\hat p$ hat einen Standardfehler $\sqrt{\hat p(1-\hat p)/n}$. Ein grobes 95-%-Intervall ist

$$\hat p \pm 1{,}96\,\sqrt{\frac{\hat p(1-\hat p)}{n}}.$$

Schreibt `konfidenzintervall(p, n)`, die `(untere, obere)` zurückgibt, begrenzt auf $[0, 1]$. In Woche 9 lernt ihr mit dem Wilson-Intervall eine bessere Variante kennen.

In [ ]:
def konfidenzintervall(p, n):
    # TODO 2
    pass

In [ ]:
check("TODO 2a p = 0,8, n = 20", lambda: all(math.isclose(a, b, abs_tol=1e-3) for a, b in zip(konfidenzintervall(0.8, 20), (0.6247, 0.9753))))
check("TODO 2b Grenzen", lambda: konfidenzintervall(1.0, 20) == (1.0, 1.0) and konfidenzintervall(0.95, 5)[1] == 1.0)

In [ ]:
ERGEBNISSE = {}

def auswerten(name, vorhersagen, kosten):
    """Vorgegeben: misst eine Stufe und speichert das Ergebnis für die Abschlusstabelle."""
    if genauigkeit(vorhersagen, TESTSET, "kategorie") is None:
        print("⏳ Erst TODO 1 lösen."); return
    werte = {f: genauigkeit(vorhersagen, TESTSET, f) for f in FELDER}
    alle = sum(1 for v, g in zip(vorhersagen, TESTSET) if v is not None and all(v.get(f) == g[f] for f in FELDER)) / len(TESTSET)
    unlesbar = sum(v is None for v in vorhersagen)
    ki = konfidenzintervall(alle, len(TESTSET)) if konfidenzintervall(0.5, 10) is not None else (float("nan"), float("nan"))
    ERGEBNISSE[name] = {**werte, "alle_drei": alle, "unlesbar": unlesbar, "kosten_cent": kosten}
    print(f"{name}: Kategorie {werte['kategorie']:.0%} | Priorität {werte['prioritaet']:.0%} | Maschine {werte['maschine']:.0%} | "
          f"alle drei richtig {alle:.0%} (95 %: {ki[0]:.0%} bis {ki[1]:.0%}) | unlesbar {unlesbar} | {kosten:.3f} Cent")

---
# Teil B · Fünf Stufen

## Stufe 1 · Der naive Prompt (vorgegeben)

Einfach fragen. Weil die Antwort freier Text ist, versucht `grob_extrahieren` die Labels mit Stichwörtern herauszufischen. Das ist absichtlich fragil.

In [ ]:
def grob_extrahieren(text):
    t = text.lower()
    def zuerst(kandidaten):
        treffer = [(t.find(stichwort), label) for stichwort, label in kandidaten if t.find(stichwort) >= 0]
        return min(treffer)[1] if treffer else None
    kategorie = zuerst([("mechan", "mechanisch"), ("elektr", "elektrisch"), ("hydraul", "hydraulik_kuehlung"),
                        ("kühl", "hydraulik_kuehlung"), ("software", "steuerung_software"), ("steuerung", "steuerung_software")])
    prioritaet = zuerst([("hoch", "hoch"), ("mittel", "mittel"), ("niedrig", "niedrig")])
    maschine = re.search(r"M-0[1-8]", text)
    return {"kategorie": kategorie, "prioritaet": prioritaet, "maschine": maschine.group(0) if maschine else None}

vorhersagen_1, kosten_1 = [], 0.0
for fall in TESTSET:
    antwort = client.chat.completions.create(model=MODEL, max_tokens=300,
                                             messages=[{"role": "user", "content": f"Klassifiziere diese Störungsmeldung: {fall['text']}"}])
    vorhersagen_1.append(grob_extrahieren(antworttext(antwort)))
    kosten_1 += kosten_cent(antwort.usage)

print("Beispielantwort:\n", antworttext(antwort)[:500], "\n")
auswerten("Stufe 1 naiv", vorhersagen_1, kosten_1)

> **Zum Nachdenken:** Warum ist die Maschinengenauigkeit wahrscheinlich nahe 0? Was weiß das Modell über die Nordwerk GmbH, und was nicht?

## Stufe 2 · Ein klarer System-Prompt

Ein guter Prompt enthält sechs Bausteine: **Rolle**, **Kontext**, **Aufgabe**, **Definitionen und Regeln**, **Ausgabeformat** und optional **Beispiele**. XML-Tags wie `<maschinen>…</maschinen>` helfen dem Modell, Teile des Prompts auseinanderzuhalten.

### TODO 3 · System-Prompt schreiben
Schreibt `SYSTEM_KLAR`. Er muss enthalten:
- eine Rolle (zum Beispiel Instandhaltungs-Assistent)
- die Maschinenliste mit IDs (Tipp: `json.dumps(MASCHINEN, ensure_ascii=False)` in einen f-String einsetzen)
- die vier erlaubten Kategorien und die drei Prioritäten **mit ihren Definitionen**
- die Anweisung, **nur** ein JSON-Objekt mit den Schlüsseln `kategorie`, `prioritaet`, `maschine` auszugeben

In [ ]:
# TODO 3
SYSTEM_KLAR = """

"""
print(SYSTEM_KLAR)

In [ ]:
check("TODO 3a alle Maschinen-IDs", lambda: all(mid in SYSTEM_KLAR for mid in MASCHINEN))
check("TODO 3b alle Kategorien und Prioritäten", lambda: all(w in SYSTEM_KLAR for w in ["mechanisch", "elektrisch", "hydraulik_kuehlung", "steuerung_software", "hoch", "mittel", "niedrig"]))
check("TODO 3c JSON-Format verlangt", lambda: "JSON" in SYSTEM_KLAR and "prioritaet" in SYSTEM_KLAR)

### TODO 4 · JSON robust lesen
Auch mit klarer Anweisung kommt manchmal Text vor oder nach dem JSON, etwa ein Markdown-Codeblock. Schreibt `parse_json(text)`:
1. Versucht `json.loads(text)`.
2. Klappt das nicht, nehmt den Abschnitt vom ersten `{` bis zum letzten `}` und versucht es erneut.
3. Klappt auch das nicht, gebt `None` zurück.

In [ ]:
def parse_json(text):
    # TODO 4
    pass

In [ ]:
check("TODO 4a reines JSON", lambda: parse_json('{"a": 1}') == {"a": 1})
check("TODO 4b JSON im Codeblock", lambda: parse_json('Hier:\n```json\n{"a": 1}\n```') == {"a": 1})
check("TODO 4c Unsinn → None", lambda: parse_json('{"a": 1}') == {"a": 1} and parse_json("keine Ahnung") is None and parse_json("{kaputt") is None)

In [ ]:
vorhersagen_2, kosten_2 = [], 0.0
for fall in (TESTSET if SYSTEM_KLAR.strip() else []):
    antwort = client.chat.completions.create(model=MODEL, max_tokens=200,
                                             messages=[{"role": "system", "content": SYSTEM_KLAR},
                                                       {"role": "user", "content": fall["text"]}])
    vorhersagen_2.append(parse_json(antworttext(antwort)))
    kosten_2 += kosten_cent(antwort.usage)
if vorhersagen_2: auswerten("Stufe 2 klar", vorhersagen_2, kosten_2)
else: print("⏳ Erst TODO 3 lösen.")

## Stufe 3 · Few-Shot: Beispiele zeigen statt beschreiben

Beispiele werden als frühere Gesprächsrunden übergeben: abwechselnd eine Nutzer-Nachricht mit der Meldung und eine Assistenten-Nachricht mit der richtigen JSON-Antwort. Danach folgt die eigentliche Meldung.

### TODO 5 · Nachrichtenliste bauen
`few_shot_nachrichten(text)` gibt eine Liste zurück: für jedes Element aus `BEISPIELE` erst `{"role": "user", "content": <text>}`, dann `{"role": "assistant", "content": <JSON mit den drei Labels>}`, am Ende die neue Meldung als `user`.

In [ ]:
def few_shot_nachrichten(text):
    nachrichten = []
    # TODO 5
    return nachrichten

In [ ]:
_n = few_shot_nachrichten("Testmeldung")
check("TODO 5a Länge 2·3 + 1", lambda: len(_n) == 7)
check("TODO 5b Rollen wechseln, Ende ist user", lambda: [x["role"] for x in _n] == ["user", "assistant"] * 3 + ["user"] and _n[-1]["content"] == "Testmeldung")
check("TODO 5c Antworten sind gültiges JSON mit drei Feldern", lambda: len(_n) == 7 and all(set(json.loads(x["content"])) == set(FELDER) for x in _n if x["role"] == "assistant"))

In [ ]:
vorhersagen_3, kosten_3 = [], 0.0
for fall in (TESTSET if few_shot_nachrichten("x") else []):
    antwort = client.chat.completions.create(model=MODEL, max_tokens=200,
                                             messages=[{"role": "system", "content": SYSTEM_KLAR}] + few_shot_nachrichten(fall["text"]))
    vorhersagen_3.append(parse_json(antworttext(antwort)))
    kosten_3 += kosten_cent(antwort.usage)
if vorhersagen_3: auswerten("Stufe 3 Few-Shot", vorhersagen_3, kosten_3)
else: print("⏳ Erst TODO 5 lösen.")

## Stufe 4 · Structured Output

Mit **Structured Outputs** schickt man ein JSON-Schema mit, und der Server beschränkt die Ausgabe auf schemakonforme Tokens (eingeschränkte Dekodierung). Die KI-Toolbox unterstützt das über `response_format`. Die Bibliothek validiert die Antwort zusätzlich. Das Schema beschreibt ihr am bequemsten als **Pydantic-Modell**. `Literal[...]` legt die erlaubten Werte fest.

### TODO 6 · Das Pydantic-Modell
Ergänzt die Felder `kategorie`, `prioritaet` und `maschine` mit `Literal` und den erlaubten Werten. Das Feld `begruendung` ist vorgegeben. Eine kurze Begründung hilft später bei der Fehlersuche.

In [ ]:
class Klassifikation(BaseModel):
    # TODO 6: kategorie, prioritaet, maschine als Literal-Felder
    begruendung: str = Field(description="Ein kurzer Satz, warum diese Einordnung passt.")

In [ ]:
def _ungueltig(daten):
    try:
        Klassifikation(**daten); return False
    except ValidationError:
        return True

_ok = {"kategorie": "elektrisch", "prioritaet": "hoch", "maschine": "M-05", "begruendung": "Stromquelle defekt."}
check("TODO 6a gültige Daten werden akzeptiert", lambda: Klassifikation(**_ok).maschine == "M-05")
check("TODO 6b falsche Kategorie wird abgelehnt", lambda: _ungueltig({**_ok, "kategorie": "pneumatisch"}))
check("TODO 6c falsche Maschine wird abgelehnt", lambda: _ungueltig({**_ok, "maschine": "M-99"}))

### TODO 7 · Mit Schema klassifizieren
Schreibt `klassifiziere_strukturiert(text, **extra)`. Sie ruft `client.chat.completions.parse(...)` auf mit `model`, `max_tokens=300`, den Nachrichten (System-Nachricht mit `SYSTEM_KLAR`, dann die Few-Shot-Nachrichten) und `response_format=Klassifikation`, und reicht `**extra` weiter. Zurück kommt die ganze Antwort. Das validierte Objekt steckt in `antwort.choices[0].message.parsed`.

In [ ]:
def klassifiziere_strukturiert(text, **extra):
    # TODO 7
    pass

In [ ]:
vorhersagen_4, kosten_4 = [], 0.0
if klassifiziere_strukturiert("Test") is None:
    print("⏳ Erst TODO 7 lösen.")
else:
    for fall in TESTSET:
        antwort = klassifiziere_strukturiert(fall["text"])
        obj = antwort.choices[0].message.parsed
        vorhersagen_4.append(obj.model_dump() if obj is not None else None)
        kosten_4 += kosten_cent(antwort.usage)
    print("Beispiel:", vorhersagen_4[0])
    auswerten("Stufe 4 Structured Output", vorhersagen_4, kosten_4)

In [ ]:
check("TODO 7 Antwort ist ein validiertes Objekt", lambda: isinstance(klassifiziere_strukturiert(TESTSET[0]["text"]).choices[0].message.parsed, Klassifikation))

## Stufe 5 · Erst überlegen, dann einordnen

Reasoning-Modelle „denken“ vor der Antwort nach, indem sie zuerst einen Gedankengang erzeugen. Das geht auch modellunabhängig: Wir lassen das Modell seine **Überlegung als erstes Feld** ins JSON schreiben. Weil Sprachmodelle Token für Token von vorn nach hinten erzeugen (Woche 5), steht die Überlegung schon da, wenn die Labels entstehen. Dafür braucht die Antwort mehr Output-Tokens.

### TODO 8 · Überlegung zuerst
1. Schreibt die Klasse `KlassifikationMitUeberlegung` mit dem Feld `ueberlegung: str` **an erster Stelle** (Beschreibung: „Kurz Schritt für Schritt: welche Maschine, welches Bauteil, wie dringend?“), danach dieselben Felder wie `Klassifikation`.
2. Passt `klassifiziere_strukturiert` so an, dass `schema` und `max_tokens` aus `extra` Vorrang haben (Tipp: `extra.pop("schema", Klassifikation)` und `extra.pop("max_tokens", 300)`).
3. Klassifiziert alle Fälle mit `schema=KlassifikationMitUeberlegung` und `max_tokens=800`.

In [ ]:
# TODO 8, Teil 1 und 2: Klasse schreiben und klassifiziere_strukturiert anpassen

vorhersagen_5, kosten_5 = [], 0.0
for fall in TESTSET:
    # TODO 8, Teil 3: antwort = klassifiziere_strukturiert(..., schema=..., max_tokens=...)
    antwort = None
    if antwort is None:
        print("⏳ Erst TODO 8 lösen."); break
    obj = antwort.choices[0].message.parsed
    vorhersagen_5.append(obj.model_dump() if obj is not None else None)
    kosten_5 += kosten_cent(antwort.usage)
if len(vorhersagen_5) == len(TESTSET):
    print("Beispiel-Überlegung:", vorhersagen_5[0]["ueberlegung"][:200])
    auswerten("Stufe 5 mit Überlegung", vorhersagen_5, kosten_5)

In [ ]:
check("TODO 8 Stufe 5 vollständig ausgewertet", lambda: "Stufe 5 mit Überlegung" in ERGEBNISSE)

## Abschlusstabelle

In [ ]:
print(f"{'Stufe':28s} {'Kat.':>5s} {'Prio.':>6s} {'Masch.':>7s} {'alle 3':>7s} {'unlesbar':>9s} {'Cent*':>7s}")
for name, r in ERGEBNISSE.items():
    print(f"{name:28s} {r['kategorie']:5.0%} {r['prioritaet']:6.0%} {r['maschine']:7.0%} {r['alle_drei']:7.0%} {r['unlesbar']:9d} {r['kosten_cent']:7.3f}")
print("* Cent: was dieselben Aufrufe bei einem kommerziellen Anbieter zum Referenzpreis kosten würden. In der KI-Toolbox zahlt ihr nichts.")

> **Zum Nachdenken:**
> - Welche Stufe brachte den größten Sprung, und bei welchem Feld?
> - Überlappen sich die Konfidenzintervalle zweier Stufen? Dann ist der Unterschied bei 20 Fällen vielleicht nur Zufall.
> - Lohnt sich die Überlegung hier? Vergleicht den Gewinn an Genauigkeit mit dem Mehr an Output-Tokens und Wartezeit.

---
## Bonus

**B1 · Fehleranalyse.** Lasst euch für die beste Stufe alle falsch klassifizierten Fälle ausgeben. Ist das Modell falsch, oder ist das Label diskutabel? (Kandidat: der flackernde Bildschirm.)

**B2 · Vorsicht beim Optimieren.** Wenn ihr den Prompt so lange anpasst, bis alle 20 Fälle stimmen, was habt ihr dann gemessen? Warum braucht man getrennte Entwicklungs- und Testdaten (Woche 9)?

**B3 · Wissensstichtag.** Fragt Mistral Small 4 nach einem Ereignis aus dem Sommer 2026. Sein Trainingswissen endet deutlich früher. Wie geht das Modell damit um? Gibt es zu, dass es das nicht wissen kann?

In [ ]:
# B1 / B3

---
# Teil C · Projektstart: Teambildung (letzte 15 Minuten)

Ab Woche 7 baut ihr in Teams einen eigenen Assistenten, der jede Woche um einen Baustein wächst. Füllt heute den **Projekt-Steckbrief** aus (Vorlage `W06_Projekt-Steckbrief.md`):

1. **Team** (zwei oder drei Personen) und **Track**:
   - A: Hochschulgruppen-Assistent
   - B: Instandhaltungs-Assistent für die Nordwerk GmbH
   - C: eigene Idee nach Absprache
2. **Drei User Stories** im Format „Als … möchte ich …, damit …“
3. **Zehn Testfragen** mit erwarteter Antwort. Daraus wird ab Woche 9 euer Testset.

Wichtig: Nur erfundene Daten verwenden, keine echten Personendaten.

## Zusammenfassung

- Assistenten entstehen in Stufen: **Pretraining** auf Text, **Supervised Fine-Tuning** auf Anweisungen, **Präferenz-Training** (RLHF, Constitutional AI) und oft **Reasoning-Training**.
- Ein guter Prompt enthält Rolle, Kontext, Aufgabe, Definitionen, Format und Beispiele. Fehlendes Wissen (Maschinenliste) kann kein Prompt-Trick ersetzen.
- **Structured Outputs** garantieren ein Schema, Pydantic validiert es in Python.
- Verbesserungen **misst** man gegen ein Testset und gibt die Unsicherheit an. Bei 20 Fällen sind die Intervalle breit.
- **Überlegung zuerst** (eine Form von Chain of Thought) kostet zusätzliche Output-Tokens. Ob es sich lohnt, entscheidet die Messung.

### Ausblick auf Woche 7
Die Maschinenliste passte noch in den Prompt. Ein 200-seitiges Wartungshandbuch nicht. Wie gibt man einem Modell **privates Wissen**, ohne es neu zu trainieren? Die Antwort heißt **RAG**.